# AIRPATH-AI Milestone 3C — target-time PM2.5 integration

This notebook connects the frozen station forecaster, IDW p=1 spatial estimator, and ordered road-segment ETAs.

Safeguards:

- non-hourly ETAs are explicitly ceiled to the next supported hour without interpolation;
- only frozen t+1h/t+2h/t+3h forecasts are accepted;
- deployment mode accepts exact pre-origin lag bundles, never future observations;
- oracle observations are kept in a separate pathway;
- outputs are segment PM2.5 estimates, not exposure or route recommendations;
- second-level ETAs do not imply minute-level PM2.5 accuracy.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.target_time_integration import (
    EXAMPLE_FORECASTING_ORIGIN,
    generate_integration_outputs,
    map_target_time,
)

In [2]:
mapping_examples = [
    map_target_time("2022-02-28 07:00:00", EXAMPLE_FORECASTING_ORIGIN),
    map_target_time("2022-02-28 06:03:00", EXAMPLE_FORECASTING_ORIGIN),
    map_target_time("2022-02-28 09:01:00", EXAMPLE_FORECASTING_ORIGIN),
]
pd.DataFrame(
    {
        "requested_target_time": [item.requested_target_time for item in mapping_examples],
        "supported_target_time": [item.supported_target_time for item in mapping_examples],
        "mapping_method": [item.mapping_method for item in mapping_examples],
        "horizon_hours": [item.forecast_horizon_hours for item in mapping_examples],
        "supported": [item.supported for item in mapping_examples],
        "status": [item.status for item in mapping_examples],
    }
)

,requested_target_time,supported_target_time,mapping_method,horizon_hours,supported,status
0,2022-02-28 07:00:00,2022-02-28 07:00:00,exact_hour,1,True,supported
1,2022-02-28 06:03:00,2022-02-28 07:00:00,ceiling_to_next_hour_no_interpolation,1,True,supported
2,2022-02-28 09:01:00,2022-02-28 10:00:00,ceiling_to_next_hour_no_interpolation,4,False,unsupported_forecast_horizon


In [3]:
outputs = generate_integration_outputs(
    network_path=PROJECT_ROOT / "data/processed/road_network/healthyair_pilot_osm.json.gz",
    clean_csv=PROJECT_ROOT / "data/processed/airquality_hcmc_clean.csv",
    forecaster_path=PROJECT_ROOT / "data/processed/models/hourly_station_forecaster.joblib",
    processed_directory=PROJECT_ROOT / "data/processed/target_time",
    report_root=PROJECT_ROOT / "reports",
)
outputs["summary"]

,pipeline_mode,travel_mode,route_id,segments,mapped_target_hours,pm25_min,pm25_mean,pm25_max,supported_reliability_segments,moderate_reliability_segments,weak_spatial_support_segments
0,oracle,walking,walking-1,121,1,15.669957,27.666542,33.258038,113,8,0
1,oracle,motorbike,motorbike-1,133,1,15.669957,26.954337,33.258038,130,3,0
2,deployment,walking,walking-1,121,1,15.673202,21.757606,25.102570,113,8,0
3,deployment,motorbike,motorbike-1,133,1,15.673202,21.389756,25.102570,130,3,0


In [4]:
sample_columns = [
    "station_value_source",
    "mode",
    "route_id",
    "segment_index",
    "requested_target_time",
    "supported_target_time",
    "forecast_horizon_hours",
    "predicted_pm25",
    "reliability_status",
]
display(outputs["sample"][sample_columns])

,station_value_source,mode,route_id,segment_index,requested_target_time,supported_target_time,forecast_horizon_hours,predicted_pm25,reliability_status
0,oracle_observed,walking,walking-1,1,2022-02-28T06:01:01.276915927,2022-02-28T07:00:00,1,33.258038,supported
1,oracle_observed,walking,walking-1,2,2022-02-28T06:02:28.376424359,2022-02-28T07:00:00,1,33.196144,supported
2,oracle_observed,walking,walking-1,3,2022-02-28T06:03:26.392246959,2022-02-28T07:00:00,1,33.146262,supported
3,oracle_observed,walking,walking-1,121,2022-02-28T06:46:13.918312141,2022-02-28T07:00:00,1,15.669957,supported
4,oracle_observed,motorbike,motorbike-1,1,2022-02-28T06:00:12.255383185,2022-02-28T07:00:00,1,33.258038,supported
5,oracle_observed,motorbike,motorbike-1,2,2022-02-28T06:00:29.675284872,2022-02-28T07:00:00,1,33.196144,supported
6,oracle_observed,motorbike,motorbike-1,3,2022-02-28T06:00:41.278449392,2022-02-28T07:00:00,1,33.146262,supported
7,oracle_observed,motorbike,motorbike-1,133,2022-02-28T06:09:12.899526607,2022-02-28T07:00:00,1,15.669957,supported
8,deployment_forecast,walking,walking-1,1,2022-02-28T06:01:01.276915927,2022-02-28T07:00:00,1,25.102570,supported
9,deployment_forecast,walking,walking-1,2,2022-02-28T06:02:28.376424359,2022-02-28T07:00:00,1,24.989151,supported


In [5]:
deployment_first = outputs["deployment_records"][0]
oracle_first = outputs["oracle_records"][0]
pd.DataFrame(
    {
        "pathway": [oracle_first.station_value_source, deployment_first.station_value_source],
        "station_values_target_time": [
            oracle_first.station_values_target_time,
            deployment_first.station_values_target_time,
        ],
        "station_values_used": [
            dict(oracle_first.station_values_used),
            dict(deployment_first.station_values_used),
        ],
        "segment_pm25": [oracle_first.predicted_pm25, deployment_first.predicted_pm25],
    }
)

,pathway,station_values_target_time,station_values_used,segment_pm25
0,oracle_observed,2022-02-28 07:00:00,"{1: 25.08333333, 2: 50.37666667, 3: 52.3233333...",33.258038
1,deployment_forecast,2022-02-28 07:00:00,"{1: 22.305646896362305, 2: 21.521059036254883,...",25.102570


## Interpretation boundary

The output is `PM2.5(segment midpoint, supported hourly target)`. For example, a 06:03 passage maps to 07:00 and records the 57-minute offset. It is not an interpolated 06:03 observation and does not establish minute-level predictive accuracy.

No segment values are summed or duration-weighted here. Exposure aggregation and route optimization remain outside Milestone 3C.